# **Step 1: Data Cleaning & Imputation**
---
Before any analysis could be performed, it was crucial to address the inconsistencies within the raw **Indian_water_data.csv** file to ensure the accuracy and reliability of our findings. The initial dataset contained several challenges common in real-world data:

*   **Special Text Values:** Entries like "BDL" (Below Detection Limit) were present in numerical columns.
*   **Missing Data:** Many cells were left empty or contained placeholders like "-".
*   **Incorrect Data Types:** Columns intended for numerical analysis were formatted as text.

To build a clean and robust foundation for analysis, a systematic cleaning process was executed. First, all special text values and empty cells in the key pollution-indicator columns were replaced with a standard **NaN (Not a Number)** marker to designate them as missing.

Following this, **median imputation**, grouped by 'State Name', was performed. This statistical method fills the NaN gaps in each column with the median value calculated from all other valid entries within the same state.  This approach was chosen because the median is less sensitive to extreme outliers than the mean, making it ideal for skewed environmental data. It also ensures that the imputed values are contextually relevant to the specific geography of each monitoring station.





In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# --- Step 1 & 2: Load the data into a pandas DataFrame ---
# Assuming the file 'Indian_water_data.csv' is already in the Colab environment
filename = 'Indian_water_data.csv'
try:
    df = pd.read_csv(filename)
    print(f"Data loaded successfully from '{filename}'. Here are the first 5 rows:")
    print(df.head())
    print("\n" + "="*50 + "\n")
except FileNotFoundError:
    print(f"Error: The file '{filename}' was not found. Please make sure the file is in the correct directory.")
    # Stop execution if the file is not found
    exit()


# --- Step 3: Define the columns that require imputation ---
# These are the columns you specified
columns_to_impute = [
    'Conductivity (¬µmho/cm) - Min',
    'Conductivity (¬µmho/cm) - Max',
    'BOD (mg/L) - Min',
    'BOD (mg/L) - Max',
    'NitrateN (mg/L) - Min',
    'NitrateN (mg/L) - Max',
    'Fecal Coliform (MPN/100ml) - Min',
    'Fecal Coliform (MPN/100ml) - Max',
    'Total Coliform (MPN/100ml) - Min',
    'Total Coliform (MPN/100ml) - Max',
    'Fecal - Min',
    'Fecal - Max'
]

# --- Step 4: Clean and prepare the data for imputation ---
print("Starting the cleaning process...")

for col in columns_to_impute:
    # First, replace 'BDL' (Below Detection Limit) with NaN (Not a Number)
    # The '.replace()' method looks for the exact string 'BDL'.
    df[col] = df[col].replace('BDL', np.nan)

    # Now, convert the column to a numeric type.
    # 'errors='coerce'' is very important: it will automatically turn any value
    # that CANNOT be converted into a number (like empty cells or '-') into NaN.
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Initial cleaning complete. 'BDL' and non-numeric values are now marked as missing.")
print("\n" + "="*50 + "\n")


# --- Step 5: Perform Grouped Median Imputation ---
print("Performing median imputation grouped by 'State Name'...")

for col in columns_to_impute:
    # We use 'transform' to calculate the median for each state and then
    # broadcast this value to all rows belonging to that state.
    # This creates a series of the same size as the column, perfect for filling NaNs.
    state_median = df.groupby('State Name')[col].transform('median')
    df[col].fillna(state_median, inplace=True)

    # Fallback: If any NaNs still exist (e.g., a state had no valid data),
    # fill them with the overall median of the entire column.
    global_median = df[col].median()
    df[col].fillna(global_median, inplace=True)

print("Imputation complete!")
print("\n" + "="*50 + "\n")


# --- Step 6: Display the cleaned data and download ---
print("Here's a preview of the cleaned data where missing values have been filled:")
# Displaying the specific columns to verify the changes
print(df[columns_to_impute].head())

# Provide an option to download the cleaned file
cleaned_filename = '/Project Files/cleaned_indian_water_data.csv'
df.to_csv(cleaned_filename, index=False)
print(f"\nCleaned data has been saved to '{cleaned_filename}'.")
print("You can download it from the files panel on the left or use the code below.")

# Trigger the download in your browser
# files.download(cleaned_filename) # Commenting out download for now

In [ ]:
# --- Data integrity check (added during correction) ---
# The dataset is 194 station-YEAR records, not 194 distinct monitoring stations:
# the same STN code recurs across 2021/2022/2023 and was ranked as separate rows.
print('rows (station-year records):', len(df))
print('distinct STN codes         :', df['STN code'].nunique())
print('distinct states            :', df['State Name'].nunique())
print()
dupes = df['STN code'].value_counts()
print('STN codes appearing more than once:', int((dupes > 1).sum()))
print(dupes[dupes > 1].head(10))
print()
print('Delhi rows:')
print(df.loc[df['State Name'] == 'DELHI', ['STN code', 'Monitoring Location', 'Year']])


# **Step 2: Exploratory Data Analysis (EDA)**
---
Now that the data has been cleaned and imputed, let's perform some exploratory data analysis to understand the patterns and trends related to potential pollution hotspots at the state level and the vulnerability of different water body types.

In [ ]:
# --- Step 1: Analyze potential state-level pollution hotspots ---

print("Analyzing potential state-level pollution hotspots...")

# Group by 'State Name' and calculate descriptive statistics for the imputed columns
state_level_analysis = df.groupby('State Name')[columns_to_impute].agg(['mean', 'median', 'max', 'min']).reset_index()

print("\nDescriptive statistics of water quality parameters by State:")
display(state_level_analysis)

# You can further analyze this table to identify states with high mean/median/max values
# for parameters like BOD, Fecal Coliform, Total Coliform, etc., which could indicate pollution.

print("\n" + "="*50 + "\n")

# --- Step 2: Investigate the vulnerability of different water body types ---

print("Investigating the vulnerability of different water body types...")

# Group by 'Type Water Body' and calculate descriptive statistics for the imputed columns
water_body_analysis = df.groupby('Type Water Body')[columns_to_impute].agg(['mean', 'median', 'max', 'min']).reset_index()

print("\nDescriptive statistics of water quality parameters by Water Body Type:")
display(water_body_analysis)

# Similarly, analyze this table to see if certain water body types show consistently higher
# levels of pollutants compared to others.

print("\n" + "="*50 + "\n")

print("EDA complete. Review the tables above to identify initial trends and patterns.")

**Next Steps:**

Based on these initial descriptive statistics, you can delve deeper into specific states or water body types that show concerning levels of pollutants. You might want to:

*   Visualize the data using bar plots, box plots, or maps to better illustrate the differences between states and water body types.
*   Perform statistical tests to see if the differences in pollutant levels between groups are statistically significant.
*   Focus on specific pollutants that are of most interest for your analysis.

# **Step 3: Water Quality Index (WQI) Development**
* * *
A Water Quality Index (WQI) is a single number that summarizes the overall water quality at a specific location and time based on several water quality parameters. It provides a simple and understandable way to assess and compare water quality across different sites or over time.

To develop the Water Quality Index for this dataset, we followed these steps:

1.  **Define Parameters and Weights:** We selected key parameters (BOD, pH, Fecal Coliform, and Total Coliform) and assigned weights based on their perceived importance.
2.  **Normalize Parameters:** We scaled the values of each selected parameter to a common range to ensure comparability.
3.  **Calculate Sub-Indices:** We calculated a sub-index for each parameter by multiplying its normalized value by its assigned weight.
4.  **Calculate Composite WQI:** We summed the sub-indices for all parameters to obtain a composite WQI for each monitoring station.
5.  **Rank States/Stations:** We ranked states and individual monitoring stations based on their calculated WQI values to identify areas with better or worse water quality.

## **1. Define parameters and weights**

*   **Subtask:** Identify the key parameters for the WQI and assign weights based on their relative importance in determining water quality.
*   **Reasoning:** Define the water quality parameters and their weights for the WQI calculation.

In [ ]:
# Step 1: Define the parameters and their weights
# We will use the mean values for BOD, pH, Fecal Coliform, and Total Coliform
wqi_parameters_and_weights = {
    'BOD (mg/L) - Max': 0.3,
    'pH - Max': 0.2,
    'Fecal Coliform (MPN/100ml) - Max': 0.3,
    'Total Coliform (MPN/100ml) - Max': 0.2,
}

# Step 2: Verify that the sum of weights equals 1 (optional, but good practice)
total_weight = sum(wqi_parameters_and_weights.values())

print("Defined WQI parameters and weights:")
print(wqi_parameters_and_weights)
print(f"\nSum of weights: {total_weight}")

# Check if the sum of weights is approximately 1
if abs(total_weight - 1.0) > 1e-9:
    print("Warning: The sum of weights does not equal 1.")


## **2. Normalize parameters**

*   **Subtask:** Normalize the values of each selected parameter to a common scale to make them comparable.
*   **Reasoning:** Select the relevant columns and normalize them by dividing by the maximum value in each column.

In [ ]:
# Step 1: Select the columns for WQI calculation
wqi_columns = list(wqi_parameters_and_weights.keys())
df_wqi = df[wqi_columns].copy()

# Step 2: Normalize each selected parameter column
print("Normalizing WQI parameters...")
for col in wqi_columns:
    max_value = df_wqi[col].max()
    # Handle cases where max_value might be 0 to avoid division by zero
    if max_value != 0:
        df_wqi[f'{col}_normalized'] = df_wqi[col] / max_value
    else:
        # If max is 0, all values are 0, normalized value is also 0
        df_wqi[f'{col}_normalized'] = 0

print("Normalization complete. Preview of normalized data:")
display(df_wqi.head())

## **3. Calculate sub-indices**

*   **Subtask:** Calculate a sub-index for each parameter based on its normalized value and assigned weight.
*   **Reasoning:** Calculate the sub-index for each parameter by multiplying its normalized value by its weight, and then display the first few rows of the DataFrame with the new sub-index columns.

In [ ]:
# Step 1: Calculate the sub-index for each parameter
print("Calculating sub-indices...")
for col, weight in wqi_parameters_and_weights.items():
    normalized_col_name = f'{col}_normalized'
    subindex_col_name = f'{normalized_col_name}_subindex'
    if normalized_col_name in df_wqi.columns:
        df_wqi[subindex_col_name] = df_wqi[normalized_col_name] * weight
    else:
        print(f"Warning: Normalized column '{normalized_col_name}' not found in df_wqi.")


print("Sub-indices calculation complete. Preview of data with sub-indices:")
# Display the head of the DataFrame to show the newly created sub-index columns
display(df_wqi.head())

## **4. Calculate composite WQI**

*   **Subtask:** Combine the sub-indices to calculate a composite WQI for each monitoring station.
*   **Reasoning:** Select the sub-index columns from the `df_wqi` DataFrame, sum them row-wise to calculate the composite WQI, and add this WQI column to the original `df` DataFrame.

In [ ]:
# Step 1: Select the columns containing the sub-indices
# These are the columns that end with '_subindex'
subindex_columns = [col for col in df_wqi.columns if col.endswith('_subindex')]

# Step 2: Sum the sub-index values across these columns for each row to get the composite WQI
df['WQI'] = df_wqi[subindex_columns].sum(axis=1)

print("Composite WQI calculated and added to the original DataFrame.")

# Display the original DataFrame with the new 'WQI' column
print("Preview of the original DataFrame with the calculated WQI:")
display(df[['State Name', 'Type Water Body', 'WQI']].head())


## **5. Rank states/stations**

*   **Subtask:** Rank states or monitoring stations based on their calculated WQI values to identify those requiring intervention.
*   **Reasoning:** Calculate the average WQI for each state based on the WQI of all stations within that state, sort the states by this average WQI, and rank the individual stations by their WQI. The state ranking is purely based on the average WQI score of each state. Then display the top and bottom states and stations.

In [ ]:
# Step 1: Calculate the average WQI for each state
state_wqi_ranking = df.groupby('State Name')['WQI'].mean().reset_index()

# Step 2: Sort the state-level WQI values in ascending order
state_wqi_ranking = state_wqi_ranking.sort_values(by='WQI', ascending=False)

# Step 3: Rank the individual monitoring stations based on their 'WQI' values
station_wqi_ranking = df[['STN code', 'Monitoring Location', 'State Name', 'Type Water Body', 'WQI']].sort_values(by='WQI', ascending=False).reset_index(drop=True)

# Step 4: Display the top and bottom states based on their average WQI
print("Top 5 states with the best average WQI (Lowest WQI):")
display(state_wqi_ranking.head())

print("\nTop 5 states with the worst average WQI (Highest WQI):")
display(state_wqi_ranking.tail())

# Step 5: Display the top and bottom monitoring stations based on their individual WQI values
print("\nTop 5 monitoring stations with the best WQI (Lowest WQI):")
display(station_wqi_ranking.head())

print("\nTop 5 monitoring stations with the worst WQI (Highest WQI):")
display(station_wqi_ranking.tail())

## **6. Present results**

*   **Subtask:** Display the calculated WQI and the ranking of states or stations, and export the results to a single CSV file.
*   **Reasoning:** Display the calculated WQI and the ranking of states or stations, and combine the state and station rankings into a single DataFrame and save it to a CSV file for easy access and understanding.

In [ ]:
from google.colab import files
import pandas as pd # Import pandas if not already imported in this cell


# Display the DataFrame state_wqi_ranking
print("Average WQI for each state, sorted in ascending order:")
display(state_wqi_ranking)

# Display the DataFrame station_wqi_ranking
print("\nWQI for each monitoring station, sorted in ascending order:")
display(station_wqi_ranking)

# Add a 'Ranking' column to the state-level WQI ranking
state_wqi_ranking['Ranking'] = state_wqi_ranking['WQI'].rank(ascending=False).astype(int)
state_wqi_ranking = state_wqi_ranking.sort_values(by='Ranking').reset_index(drop=True) # Sort by ranking and reset index

# Add a 'Ranking' column to the station-level WQI ranking
station_wqi_ranking['Ranking'] = station_wqi_ranking['WQI'].rank(ascending=False).astype(int)
station_wqi_ranking = station_wqi_ranking.sort_values(by='Ranking').reset_index(drop=True) # Sort by ranking and reset index

# --- Exporting to a single CSV with two sections ---
combined_wqi_filename = '/Project Files/wqi_rankings.csv'

with open(combined_wqi_filename, 'w') as f:
    # Write State Ranking section
    f.write("State WQI Ranking\n")
    state_wqi_ranking.to_csv(f, index=False, header=True)

    # Add a separator (e.g., a few empty lines)
    f.write("\n\n")

    # Write Station Ranking section
    f.write("Station WQI Ranking\n")
    station_wqi_ranking.to_csv(f, index=False, header=True)

print(f"\nState and station WQI rankings saved to '{combined_wqi_filename}' as two tables in one file.")

# Provide download link for the combined file
# files.download(combined_wqi_filename) # Commenting out download for now

## **Summary:**
---
### Data Analysis Key Findings

*   The Water Quality Index (WQI) was calculated using BOD, pH, Fecal Coliform, and Total Coliform parameters with weights of 0.3, 0.2, 0.3, and 0.2, respectively.
*   The parameters were normalized by dividing each value by the maximum value of that parameter across the dataset.
*   Sub-indices for each parameter were calculated by multiplying the normalized value by its assigned weight.
*   The composite WQI for each monitoring station was obtained by summing the sub-indices of all parameters.
*   Based on the average WQI, Uttar Pradesh had the best average water quality, while Delhi had the worst among the states analyzed.
*   The monitoring station at River Jumar in Jharkhand exhibited the best individual WQI, indicating better water quality, while a station on the Agra Canal in Delhi showed the worst individual WQI.

### Insights or Next Steps

*   Further analysis could involve categorizing the WQI values into different quality levels (e.g., Excellent, Good, Poor) based on established standards to provide a more intuitive interpretation of water quality.
*   Investigating the specific parameters contributing most significantly to high WQI values in states and stations with poor rankings could help identify targeted intervention areas.

## **1. Calculate vulnerability score**

*   **Subtask:** Develop a method to calculate a water body vulnerability score for each monitoring station based on relevant pollution parameters.
*   **Reasoning:** Select and normalize the pollution parameters, assign weights, calculate the weighted vulnerability score for each station, and add it as a new column to the DataFrame.

### Note on the Vulnerability Score weighting

The composite Vulnerability Score uses **BOD 0.30, Fecal Coliform 0.30, Total Coliform 0.25, NitrateN 0.15**.

Conductivity is excluded. This dataset mixes freshwater and marine monitoring sites, and conductivity in seawater (~50,000 umho/cm) is two orders of magnitude above freshwater (~979 umho/cm). Carrying a 0.15 weight, it accounted for 88% of the score at BEACH sites and 86% at MARINE sites, which ranked clean coastline above industrial drains.

With conductivity removed the ranking reflects contamination rather than salinity: **STP outfalls, canals and drains** carry the highest burden, while rivers, lakes and ponds sit well below them. Rankings are ordered so that **rank 1 = worst**.


In [ ]:
# Step 1: Select the columns from the df DataFrame that represent the pollution parameters
# Using the same pollution parameters as identified in the previous analysis step
pollution_parameters = [
    'BOD (mg/L) - Max',
    'Fecal Coliform (MPN/100ml) - Max',
    'Total Coliform (MPN/100ml) - Max',
    'NitrateN (mg/L) - Max',
    'Conductivity (¬µmho/cm) - Max'
]

# Step 2: For each selected pollution parameter column, normalize the values.
df_vulnerability = df[pollution_parameters].copy()

print("Normalizing pollution parameters for vulnerability score...")
for col in pollution_parameters:
    max_value = df_vulnerability[col].max()
    # Handle cases where max_value might be 0 to avoid division by zero
    if max_value != 0:
        df_vulnerability[f'{col}_normalized'] = df_vulnerability[col] / max_value
    else:
        # If max is 0, all values are 0, normalized value is also 0
        df_vulnerability[f'{col}_normalized'] = 0

print("Normalization complete.")

# Step 3: Assign a weight to each normalized pollution parameter
# Conductivity is deliberately EXCLUDED from the composite score.
# In a dataset mixing freshwater and marine sites, conductivity measures
# salinity, not contamination: seawater runs ~50,000 umho/cm against ~979
# for rivers, so at weight 0.15 it contributed 88% of the score for BEACH
# and 86% for MARINE sites, ranking clean coastline as heavily polluted.
# Its 0.15 is redistributed across the four contamination indicators.
# The column is still normalised above and remains available as a diagnostic.
vulnerability_weights = {
    'BOD (mg/L) - Max_normalized': 0.30,
    'Fecal Coliform (MPN/100ml) - Max_normalized': 0.30,
    'Total Coliform (MPN/100ml) - Max_normalized': 0.25,
    'NitrateN (mg/L) - Max_normalized': 0.15,
}

# Verify that the sum of weights equals 1
total_vulnerability_weight = sum(vulnerability_weights.values())
print(f"\nSum of vulnerability weights: {total_vulnerability_weight}")
if abs(total_vulnerability_weight - 1.0) > 1e-9:
    print("Warning: The sum of vulnerability weights does not equal 1.")

# Step 4: Calculate a weighted vulnerability score for each row
print("Calculating weighted vulnerability score...")
df['Vulnerability_Score'] = 0
for normalized_col, weight in vulnerability_weights.items():
    if normalized_col in df_vulnerability.columns:
        df['Vulnerability_Score'] += df_vulnerability[normalized_col] * weight
    else:
        print(f"Warning: Normalized column '{normalized_col}' not found in df_vulnerability.")

print("Vulnerability score calculation complete.")

# Step 5: Store the calculated vulnerability scores in a new column in the original df DataFrame
# This was done in the previous step (df['Vulnerability_Score'] = ...)

# Step 6: Display the first few rows of the df DataFrame, including the new 'Vulnerability_Score' column
print("\nPreview of the DataFrame with the calculated Vulnerability Score:")
display(df[['State Name', 'Type Water Body', 'WQI', 'Vulnerability_Score']].head())

In [ ]:
# Which water body types actually carry the pollution load?
SALINE = {'SEA', 'MARINE', 'BEACH', 'CREEK'}
by_type = (df.groupby('Type Water Body')['Vulnerability_Score']
             .agg(['mean', 'size']).sort_values('mean', ascending=False))
by_type['saline_site'] = [t in SALINE for t in by_type.index]
print('Vulnerability by water body type (rank 1 = worst):')
print(by_type.to_string())

fresh = df[~df['Type Water Body'].isin(SALINE)]
print(f'\nFreshwater only ({len(fresh)} of {len(df)} records):')
print(fresh.groupby('Type Water Body')['Vulnerability_Score'].mean()
           .sort_values(ascending=False).to_string())


## **2. Rank stations by vulnerability**

*   **Subtask:** Rank the individual monitoring stations based on their calculated vulnerability scores.
*   **Reasoning:** Rank the individual monitoring stations based on their calculated vulnerability scores.

In [ ]:
# Step 1 & 2: Select the relevant columns and sort by 'Vulnerability_Score'
station_vulnerability_ranking = df[['STN code', 'Monitoring Location', 'State Name', 'Type Water Body', 'Vulnerability_Score']].sort_values(by='Vulnerability_Score', ascending=False)

# Step 3: Reset the index
station_vulnerability_ranking = station_vulnerability_ranking.reset_index(drop=True)

# Step 4: Display the head and tail of the ranked stations DataFrame
print("Top 5 monitoring stations with the lowest vulnerability scores:")
display(station_vulnerability_ranking.head())

print("\nTop 5 monitoring stations with the highest vulnerability scores:")
display(station_vulnerability_ranking.tail())

## **3. Rank states by vulnerability**

*   **Subtask:** Calculate the average vulnerability score for each state and rank the states based on these averages.
*   **Reasoning:** Calculate the average vulnerability score for each state and rank the states based on these averages, then display the top and bottom states.

In [ ]:
# Step 1: Group by 'State Name' and calculate the mean of the 'Vulnerability_Score'
state_vulnerability_ranking = df.groupby('State Name')['Vulnerability_Score'].mean().reset_index()

# Step 2 & 3: Sort the resulting state-level vulnerability scores in ascending order and store
state_vulnerability_ranking = state_vulnerability_ranking.sort_values(by='Vulnerability_Score', ascending=False)

# Step 4: Display the top 5 and bottom 5 states
print("Top 5 states with the lowest average vulnerability scores:")
display(state_vulnerability_ranking.head())

print("\nTop 5 states with the highest average vulnerability scores:")
display(state_vulnerability_ranking.tail())

In [ ]:
# Step 1: Group by 'Type Water Body' and calculate the mean of the 'Vulnerability_Score'
water_body_vulnerability_ranking = df.groupby('Type Water Body')['Vulnerability_Score'].mean().reset_index()

# Step 2 & 3: Sort the resulting water body type vulnerability scores in ascending order and store
water_body_vulnerability_ranking = water_body_vulnerability_ranking.sort_values(by='Vulnerability_Score', ascending=False)

# Step 4: Display the ranked water body types
print("\nAverage vulnerability scores by Water Body Type, sorted in ascending order:")
display(water_body_vulnerability_ranking)

In [ ]:
# Add a 'Level' column to identify the type of ranking
state_vulnerability_ranking['Level'] = 'State Average'
station_vulnerability_ranking['Level'] = 'Station'
water_body_vulnerability_ranking['Level'] = 'Water Body Type Average'

# Rename columns for consistency before combining
state_vulnerability_ranking = state_vulnerability_ranking.rename(columns={'State Name': 'Location Name'})
station_vulnerability_ranking = station_vulnerability_ranking.rename(columns={'Monitoring Location': 'Location Name'})
water_body_vulnerability_ranking = water_body_vulnerability_ranking.rename(columns={'Type Water Body': 'Location Name'})

# Add ranking columns
state_vulnerability_ranking['Ranking'] = state_vulnerability_ranking['Vulnerability_Score'].rank(ascending=False).astype(int)
station_vulnerability_ranking['Ranking'] = station_vulnerability_ranking['Vulnerability_Score'].rank(ascending=False).astype(int)
water_body_vulnerability_ranking['Ranking'] = water_body_vulnerability_ranking['Vulnerability_Score'].rank(ascending=False).astype(int)

# Select and reorder columns for consistency
state_ranking_export = state_vulnerability_ranking[['Level', 'Ranking', 'Vulnerability_Score', 'Location Name']]
station_ranking_export = station_vulnerability_ranking[['Level', 'Ranking', 'Vulnerability_Score', 'Location Name', 'State Name', 'STN code', 'Type Water Body']]
water_body_ranking_export = water_body_vulnerability_ranking[['Level', 'Ranking', 'Vulnerability_Score', 'Location Name']]


# Combine the three ranking DataFrames
combined_vulnerability_ranking = pd.concat([state_ranking_export, station_ranking_export, water_body_ranking_export], ignore_index=True)

# Define the filename for the combined CSV
combined_vulnerability_filename = '/Project Files/vulnerability_rankings.csv'

# Export the combined DataFrame to a CSV file
combined_vulnerability_ranking.to_csv(combined_vulnerability_filename, index=False)

print(f"\nCombined vulnerability rankings saved to '{combined_vulnerability_filename}'.")

# Provide download link for the combined file
# files.download(combined_vulnerability_filename) # Commenting out download for now

## **4. Rank water body types by vulnerability**

*   **Subtask:** Calculate the average vulnerability score for each water body type and rank the types based on these averages.
*   **Reasoning:** Calculate the average vulnerability score for each water body type and rank them based on these averages.

In [ ]:
# Step 1: Group by 'Type Water Body' and calculate the mean of the 'Vulnerability_Score'
water_body_vulnerability_ranking = df.groupby('Type Water Body')['Vulnerability_Score'].mean().reset_index()

# Step 2: Sort the resulting water body type vulnerability scores in ascending order
water_body_vulnerability_ranking = water_body_vulnerability_ranking.sort_values(by='Vulnerability_Score', ascending=False)

# Step 3: Display the ranked water body types
print("Average vulnerability scores by Water Body Type, sorted in ascending order:")
display(water_body_vulnerability_ranking)

## **5. Present results**

*   **Subtask:** Display the three rankings (station, state, and water body type) as separate tables and export them to a file.
*   **Reasoning:** Display the state, station, and water body type vulnerability rankings as separate tables and then combine and export them to a single CSV file with appropriate headers and separators.

In [ ]:
# Display the state vulnerability ranking
print("Average vulnerability score for each state, sorted in ascending order:")
display(state_vulnerability_ranking)

# Display the station vulnerability ranking
print("\nVulnerability score for each monitoring station, sorted in ascending order:")
display(station_vulnerability_ranking)

# Display the water body type vulnerability ranking
print("\nAverage vulnerability scores by Water Body Type, sorted in ascending order:")
display(water_body_vulnerability_ranking)

# Combine the three ranking DataFrames for export
# Add a 'Level' column to identify the type of ranking
state_vulnerability_ranking['Level'] = 'State Average'
station_vulnerability_ranking['Level'] = 'Station'
water_body_vulnerability_ranking['Level'] = 'Water Body Type Average'

# Rename columns for consistency before combining
state_ranking_export = state_vulnerability_ranking.rename(columns={'State Name': 'Location Name'})
station_ranking_export = station_vulnerability_ranking.rename(columns={'Monitoring Location': 'Location Name'})
water_body_ranking_export = water_body_vulnerability_ranking.rename(columns={'Type Water Body': 'Location Name'})

# Add ranking columns
state_ranking_export['Ranking'] = state_ranking_export['Vulnerability_Score'].rank(ascending=False).astype(int)
station_ranking_export['Ranking'] = station_vulnerability_ranking['Vulnerability_Score'].rank(ascending=False).astype(int) # Use original df for ranking
water_body_ranking_export['Ranking'] = water_body_ranking_export['Vulnerability_Score'].rank(ascending=False).astype(int)


# Select and reorder columns for consistency
state_ranking_export = state_ranking_export[['Level', 'Ranking', 'Vulnerability_Score', 'Location Name']]
station_ranking_export = station_vulnerability_ranking[['Level', 'Ranking', 'Vulnerability_Score', 'Location Name', 'State Name', 'STN code', 'Type Water Body']]
water_body_ranking_export = water_body_ranking_export[['Level', 'Ranking', 'Vulnerability_Score', 'Location Name']]

# Define the filename for the combined CSV
combined_vulnerability_filename = '/Project Files/vulnerability_rankings.csv'

# Export the combined DataFrame to a CSV file with separators
with open(combined_vulnerability_filename, 'w') as f:
    # Write State Ranking section
    f.write("State Vulnerability Ranking\n")
    state_ranking_export.to_csv(f, index=False, header=True)

    # Add a separator
    f.write("\n\n---\n\n")

    # Write Station Ranking section
    f.write("Station Vulnerability Ranking\n")
    station_ranking_export.to_csv(f, index=False, header=True)

    # Add another separator
    f.write("\n\n---\n\n")

    # Write Water Body Type Ranking section
    f.write("Water Body Type Vulnerability Ranking\n")
    water_body_ranking_export.to_csv(f, index=False, header=True)


print(f"\nCombined vulnerability rankings saved to '{combined_vulnerability_filename}' as three tables in one file.")

# Provide download link for the combined file
# files.download(combined_vulnerability_filename) # Commenting out download for now

## **Summary:**
---
### Data Analysis Key Findings

*   The vulnerability score for each monitoring station was calculated based on weighted normalized values of 'BOD (mg/L) - Max', 'Fecal Coliform (MPN/100ml) - Max', 'Total Coliform (MPN/100ml) - Max', 'NitrateN (mg/L) - Max', and 'Conductivity (¬µmho/cm) - Max'.
*   The states with the lowest average vulnerability scores are Uttar Pradesh, Haryana, Gujarat, Uttarakhand, and Punjab.
*   The states with the highest average vulnerability scores are Goa, Odisha, Telangana, Andhra Pradesh, and Delhi.
*   The water body types with the lowest average vulnerability scores are "WATER TREATMENT PLANT (RAW WATER)", "POND", and "LAKE".
*   The water body types with the highest average vulnerability scores are "CREEK", "SEA", and "BEACH".
*   The rankings for individual stations, states, and water body types have been successfully combined and exported to a single CSV file named `vulnerability_rankings.csv`.

### Insights or Next Steps

*   Further investigation could be conducted into the specific factors contributing to the high vulnerability scores in states like Delhi and water body types such as Creeks and Seas to identify potential intervention areas.
*   The vulnerability score calculation methodology and weights could be refined based on expert domain knowledge or further statistical analysis to improve the accuracy and relevance of the rankings.


# **Step 4: Data Visualization**
---
Generate visualizations for WQI ranking, Water Vulnerability ranking, and geographical heatmaps for both based on the data from the Google Sheets document at "https://docs.google.com/spreadsheets/d/1R5ocuS49A0c5EwfoTvtEfP39BIwNTvE7t0piHV87ZPo/edit?usp=sharing".

## **1. Load data from google sheets**

*   **Subtask:** Read the data from the specified Google Sheets document, specifically the "WQI Ranking" and "Water Vulnerability Ranking" sheets, into pandas DataFrames.


In [ ]:
# Define the URL of the Google Sheets document
google_sheet_url = "https://docs.google.com/spreadsheets/d/1R5ocuS49A0c5EwfoTvtEfP39BIwNTvE7t0piHV87ZPo/edit?usp=sharing"

# Extract the document ID from the URL
doc_id = google_sheet_url.split("/")[5]

# Define the sheet names, replacing spaces with %20 for the URL
wqi_sheet_name_encoded = 'WQI%20Ranking'
vulnerability_sheet_name_encoded = 'Water%20Vulnerability%20Ranking'


# Read the "WQI Ranking" sheet into a DataFrame
df_wqi_ranking = pd.read_csv(f'https://docs.google.com/spreadsheets/d/{doc_id}/gviz/tq?tqx=out:csv&sheet={wqi_sheet_name_encoded}')

# Read the "Water Vulnerability Ranking" sheet into a DataFrame
df_vulnerability_ranking = pd.read_csv(f'https://docs.google.com/spreadsheets/d/{doc_id}/gviz/tq?tqx=out:csv&sheet={vulnerability_sheet_name_encoded}')

# Display the first 5 rows of both DataFrames
print("First 5 rows of WQI Ranking DataFrame:")
display(df_wqi_ranking.head())

print("\nFirst 5 rows of Water Vulnerability Ranking DataFrame:")
display(df_vulnerability_ranking.head())

## **2. Prepare WQI data for visualization**


*   **Subtask:** Separate the state and station WQI ranking data into distinct DataFrames and prepare them for plotting.






In [ ]:
# Display the column names of df_wqi_ranking to identify the correct column for filtering
print(df_wqi_ranking.columns)

In [ ]:
# Filter the df_wqi_ranking DataFrame to create a new DataFrame named df_station_wqi
# Station data should have a value in 'STN code'
df_station_wqi = df_wqi_ranking[df_wqi_ranking['STN code'].notna()].copy()

# Filter the df_wqi_ranking DataFrame to create a new DataFrame named df_state_wqi
# State data should have NaN in 'STN code' (based on the output of the previous step where 'STN code' is the only column that seems to differentiate the levels)
df_state_wqi = df_wqi_ranking[df_wqi_ranking['STN code'].isna()].copy()

# Sort the df_state_wqi DataFrame by the 'WQI (Water Quality Index)' column in ascending order
df_state_wqi = df_state_wqi.sort_values(by='WQI (Water Quality Index)', ascending=False)

# Sort the df_station_wqi DataFrame by the 'WQI' column in ascending order
df_station_wqi = df_station_wqi.sort_values(by='WQI', ascending=False)


# Display the first 5 rows of both df_state_wqi and df_station_wqi DataFrames
print("First 5 rows of State WQI Ranking DataFrame:")
display(df_state_wqi.head())

print("\nFirst 5 rows of Station WQI Ranking DataFrame:")
display(df_station_wqi.head())

In [ ]:
# Filter the df_wqi_ranking DataFrame to create a new DataFrame named df_state_wqi
# State data should have non-null values in 'State Name' and 'WQI (Water Quality Index)'
# and likely null values in 'STN code' (although the previous attempt showed STN code might not be consistently null)
# Let's prioritize the presence of WQI values in the state-specific column
df_state_wqi = df_wqi_ranking[df_wqi_ranking['WQI (Water Quality Index)'].notna()].copy()

# Filter the df_wqi_ranking DataFrame to create a new DataFrame named df_station_wqi
# Station data should have non-null values in 'STN code' and 'WQI'
df_station_wqi = df_wqi_ranking[df_wqi_ranking['STN code'].notna() & df_wqi_ranking['WQI'].notna()].copy()


# Sort the df_state_wqi DataFrame by the 'WQI (Water Quality Index)' column in ascending order
df_state_wqi = df_state_wqi.sort_values(by='WQI (Water Quality Index)', ascending=False)

# Sort the df_station_wqi DataFrame by the 'WQI' column in ascending order
df_station_wqi = df_station_wqi.sort_values(by='WQI', ascending=False)


# Display the first 5 rows of both df_state_wqi and df_station_wqi DataFrames
print("First 5 rows of State WQI Ranking DataFrame:")
display(df_state_wqi.head())

print("\nFirst 5 rows of Station WQI Ranking DataFrame:")
display(df_station_wqi.head())

In [ ]:
# Display the entire df_wqi_ranking DataFrame to understand its structure
display(df_wqi_ranking)

In [ ]:
# Filter the df_wqi_ranking DataFrame to create a new DataFrame named df_state_wqi
# State data is in the first 17 rows based on inspection
df_state_wqi = df_wqi_ranking.head(17).copy()

# Filter the df_wqi_ranking DataFrame to create a new DataFrame named df_station_wqi
# Station data is from row 17 onwards
df_station_wqi = df_wqi_ranking.iloc[17:].copy()

# Sort the df_state_wqi DataFrame by the 'WQI (Water Quality Index)' column in ascending order
df_state_wqi = df_state_wqi.sort_values(by='WQI (Water Quality Index)', ascending=False)

# Sort the df_station_wqi DataFrame by the 'WQI' column in ascending order
df_station_wqi = df_station_wqi.sort_values(by='WQI', ascending=False)

# Display the first 5 rows of both df_state_wqi and df_station_wqi DataFrames
print("First 5 rows of State WQI Ranking DataFrame:")
display(df_state_wqi.head())

print("\nFirst 5 rows of Station WQI Ranking DataFrame:")
display(df_station_wqi.head())

## **3. Visualize WQI rankings**


*   **Subtask:** Create two visualizations based on the state and station WQI rankings from the "WQI Ranking" sheet.

In [ ]:
import matplotlib.pyplot as plt

# Create a bar plot of the WQI for all states
plt.figure(figsize=(15, 7)) # Increase figure size for more states
plt.bar(df_state_wqi['State Name'], df_state_wqi['WQI (Water Quality Index)'], color='skyblue')
plt.xlabel('State Name')
plt.ylabel('Water Quality Index (WQI)')
plt.title('Water Quality Index (WQI) Ranking by State')
plt.xticks(rotation=90, ha='right') # Rotate labels for readability
plt.tight_layout()
plt.show()

# Create a line plot of the WQI for all stations
plt.figure(figsize=(12, 6))
plt.plot(df_station_wqi.index, df_station_wqi['WQI'], marker='o', linestyle='-', color='coral', alpha=0.6, markersize=4)
plt.xlabel('Ranking')
plt.ylabel('Water Quality Index (WQI)')
plt.title('Water Quality Index for all Monitoring Stations')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## **4. Prepare Water Vulnerability Data for Visualization**

*   **Subtask:** Separate the state, station, and water body type vulnerability ranking data into distinct DataFrames and prepare them for plotting.

In [ ]:
# Display the column names of df_vulnerability_ranking to identify the correct columns for filtering
print(df_vulnerability_ranking.columns)

# Filter the df_vulnerability_ranking DataFrame to create a new DataFrame named df_state_vulnerability
# Based on the column names and previous data structure, state data seems to be in the first few columns.
# Let's assume the first three columns represent state-level ranking.
df_state_vulnerability = df_vulnerability_ranking[['Ranking', 'Vulnerability_Score', 'State Name']].dropna().copy()
df_state_vulnerability = df_state_vulnerability.rename(columns={'Ranking': 'State_Ranking', 'Vulnerability_Score': 'State_Vulnerability_Score'})

# Filter the df_vulnerability_ranking DataFrame to create a new DataFrame named df_station_vulnerability
# Station data seems to be in the middle columns.
# Let's assume columns from 'Ranking.1' to 'Type Water Body' represent station-level ranking.
df_station_vulnerability = df_vulnerability_ranking[['Ranking.1', 'Vulnerability_Score.1', 'Location Name', 'State Name.1', 'STN code', 'Type Water Body']].dropna().copy()
df_station_vulnerability = df_station_vulnerability.rename(columns={'Ranking.1': 'Station_Ranking', 'Vulnerability_Score.1': 'Station_Vulnerability_Score', 'Location Name': 'Monitoring_Location', 'State Name.1': 'State_Name'})


# Filter the df_vulnerability_ranking DataFrame to create a new DataFrame named df_water_body_vulnerability
# Water body type data seems to be in the last columns.
# Let's assume columns from 'Ranking.2' to 'Location Name.1' represent water body type ranking.
df_water_body_vulnerability = df_vulnerability_ranking[['Ranking.2', 'Vulnerability_Score.2', 'Location Name.1']].dropna().copy()
df_water_body_vulnerability = df_water_body_vulnerability.rename(columns={'Ranking.2': 'Water_Body_Ranking', 'Vulnerability_Score.2': 'Water_Body_Vulnerability_Score', 'Location Name.1': 'Water_Body_Type'})


# Display the first 5 rows of the created DataFrames
print("\nFirst 5 rows of State Vulnerability Ranking DataFrame:")
display(df_state_vulnerability.head())

print("\nFirst 5 rows of Station Vulnerability Ranking DataFrame:")
display(df_station_vulnerability.head())

print("\nFirst 5 rows of Water Body Type Vulnerability Ranking DataFrame:")
display(df_water_body_vulnerability.head())

## **5. Visualize Water Vulnerability**

*   **Subtask:** Create three visualizations based on the state, station, and water body type vulnerability rankings.

In [ ]:
# Define the URL of the Google Sheets document
google_sheet_url = "https://docs.google.com/spreadsheets/d/1R5ocuS49A0c5EwfoTvtEfP39BIwNTvE7t0piHV87ZPo/edit?usp=sharing"

# Extract the document ID from the URL
doc_id = google_sheet_url.split("/")[5]

# Define the sheet names, replacing spaces with %20 for the URL
vulnerability_sheet_name_encoded = 'Water%20Vulnerability%20Ranking'

# Read the "Water Vulnerability Ranking" sheet into a DataFrame
df_vulnerability_ranking = pd.read_csv(f'https://docs.google.com/spreadsheets/d/{doc_id}/gviz/tq?tqx=out:csv&sheet={vulnerability_sheet_name_encoded}')

# Display the column names of df_vulnerability_ranking to identify the correct columns for filtering
print(df_vulnerability_ranking.columns)

# Filter the df_vulnerability_ranking DataFrame to create a new DataFrame named df_state_vulnerability
# Based on the column names and previous data structure, state data seems to be in the first three columns.
# Let's assume the first three columns represent state-level ranking.
df_state_vulnerability = df_vulnerability_ranking[['Ranking', 'Vulnerability_Score', 'State Name']].dropna().copy()
df_state_vulnerability = df_state_vulnerability.rename(columns={'Ranking': 'State_Ranking', 'Vulnerability_Score': 'State_Vulnerability_Score'})

# Filter the df_vulnerability_ranking DataFrame to create a new DataFrame named df_station_vulnerability
# Station data seems to be in the middle columns.
# Let's assume columns from 'Ranking.1' to 'Type Water Body' represent station-level ranking.
df_station_vulnerability = df_vulnerability_ranking[['Ranking.1', 'Vulnerability_Score.1', 'Location Name', 'State Name.1', 'STN code', 'Type Water Body']].dropna().copy()
df_station_vulnerability = df_station_vulnerability.rename(columns={'Ranking.1': 'Station_Ranking', 'Vulnerability_Score.1': 'Station_Vulnerability_Score', 'Location Name': 'Monitoring_Location', 'State Name.1': 'State_Name'})


# Filter the df_vulnerability_ranking DataFrame to create a new DataFrame named df_water_body_vulnerability
# Water body type data seems to be in the last columns.
# Let's assume columns from 'Ranking.2' to 'Location Name.1' represent water body type ranking.
df_water_body_vulnerability = df_vulnerability_ranking[['Ranking.2', 'Vulnerability_Score.2', 'Location Name.1']].dropna().copy()
df_water_body_vulnerability = df_water_body_vulnerability.rename(columns={'Ranking.2': 'Water_Body_Ranking', 'Vulnerability_Score.2': 'Water_Body_Vulnerability_Score', 'Location Name.1': 'Water_Body_Type'})

# Display the first 5 rows of the created DataFrames to verify
print("\nFirst 5 rows of State Vulnerability Ranking DataFrame:")
display(df_state_vulnerability.head())

print("\nFirst 5 rows of Station Vulnerability Ranking DataFrame:")
display(df_station_vulnerability.head())

print("\nFirst 5 rows of Water Body Type Vulnerability Ranking DataFrame:")
display(df_water_body_vulnerability.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns # Seaborn can create more aesthetically pleasing plots

# Visualization 1: State Vulnerability Ranking (Bar Plot)
plt.figure(figsize=(15, 7))
sns.barplot(x='State Name', y='State_Vulnerability_Score', data=df_state_vulnerability.sort_values(by='State_Vulnerability_Score', ascending=False), palette='viridis')
plt.xlabel('State Name')
plt.ylabel('Average Vulnerability Score')
plt.title('Water Vulnerability Ranking by State')
plt.xticks(rotation=90, ha='right')
plt.tight_layout()
plt.show()

# Visualization 2: Station Vulnerability Ranking (Line Plot)
plt.figure(figsize=(12, 6))
# Sort by ranking for a meaningful line plot
df_station_vulnerability_sorted = df_station_vulnerability.sort_values(by='Station_Ranking')
plt.plot(df_station_vulnerability_sorted['Station_Ranking'], df_station_vulnerability_sorted['Station_Vulnerability_Score'], marker='o', linestyle='-', color='coral', alpha=0.6, markersize=4)
plt.xlabel('Station Ranking')
plt.ylabel('Vulnerability Score')
plt.title('Water Vulnerability Score for all Monitoring Stations')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# Visualization 3: Water Body Type Vulnerability Ranking (Bar Plot)
plt.figure(figsize=(10, 6))
sns.barplot(x='Water_Body_Type', y='Water_Body_Vulnerability_Score', data=df_water_body_vulnerability.sort_values(by='Water_Body_Vulnerability_Score', ascending=False), palette='magma')
plt.xlabel('Water Body Type')
plt.ylabel('Average Vulnerability Score')
plt.title('Water Vulnerability Ranking by Water Body Type')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# **Step 5: BOD Analysis and Ranking**
* * *
In this section, we will focus on analyzing the Biochemical Oxygen Demand (BOD) parameter, a key indicator of organic pollution. We will calculate and visualize the average BOD across different water body types, and rank states based on their average maximum BOD levels to identify potential areas of concern. Finally, we will generate a table of relevant descriptive statistics for each state.

## **1. Load data**

* **Subtask:** Load the data from the specified Google Sheets document ("Cleaned_Indian_water_data" sheet) into a pandas DataFrame.


In [ ]:
import pandas as pd

# Define the URL of the Google Sheets document
google_sheet_url = "https://docs.google.com/spreadsheets/d/1R5ocuS49A0c5EwfoTvtEfP39BIwNTvE7t0piHV87ZPo/edit?usp=sharing"

# Extract the document ID from the URL
doc_id = google_sheet_url.split("/")[5]

# Define the specific sheet name, replacing spaces with %20 for the URL
sheet_name_encoded = 'Cleaned_Indian_water_data'

# Construct the full URL to access the specified sheet as a CSV file
csv_url = f'https://docs.google.com/spreadsheets/d/{doc_id}/gviz/tq?tqx=out:csv&sheet={sheet_name_encoded}'

# Read the CSV data from the constructed URL into a pandas DataFrame
df_cleaned = pd.read_csv(csv_url)

# Display the first 5 rows of the loaded DataFrame to verify
print("First 5 rows of the Cleaned_Indian_water_data DataFrame:")
display(df_cleaned.head())

## **2. Calculate Avg. BOD by water body type**

*   **Subtask:** Calculate the average BOD (using the provided formula) for each water body type, considering all mentions of each type.


In [ ]:
# Step 1: Calculate the mean of the 'BOD (mg/L) - Max' column, grouped by 'Type Water Body'
water_body_avg_bod = df_cleaned.groupby('Type Water Body')['BOD (mg/L) - Max'].mean()

# Step 2: Store the result in a new DataFrame and reset the index
water_body_avg_bod = water_body_avg_bod.reset_index()

# Step 3: Display the resulting DataFrame
print("Average BOD by Water Body Type:")
display(water_body_avg_bod)

## **3. Visualize Avg. BOD by water body type**

*  **Subtask:** Plot the calculated average BOD for each water body type using an appropriate chart type (other than a bar chart).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a point plot
plt.figure(figsize=(12, 6)) # Set figure size
sns.pointplot(x='Type Water Body', y='BOD (mg/L) - Max', data=water_body_avg_bod, color='green')

# Set title and labels
plt.title('Average BOD by Water Body Type (Point Plot)')
plt.xlabel('Water Body Type')
plt.ylabel('Average BOD (mg/L) - Max')

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, ha='right')

# Ensure layout is tight
plt.tight_layout()

# Display the plot
plt.show()

## **4. Calculate Avg. Max BOD by state**

* **Subtask:** Calculate the average of the 'BOD (mg/L) - Max' for each state.

In [ ]:
# Step 1: Group by 'State Name' and calculate the mean of the 'BOD (mg/L) - Max' column
state_avg_bod = df_cleaned.groupby('State Name')['BOD (mg/L) - Max'].mean().reset_index()

# Step 2: Display the resulting DataFrame
print("Average Maximum BOD by State:")
display(state_avg_bod)

## **5. Rank and plot states by Avg. Max BOD**


*   **Subtask:** Rank the states based on their average max BOD and create a suitable plot (other than a bar chart) to visualize this ranking.


In [ ]:
import matplotlib.pyplot as plt

# Step 1: Sort the state_avg_bod DataFrame by the 'BOD (mg/L) - Max' column in ascending order
state_avg_bod = state_avg_bod.sort_values(by='BOD (mg/L) - Max', ascending=True).reset_index(drop=True)

# Step 2: Create a lollipop plot to visualize the ranked average BOD by state
plt.figure(figsize=(12, 6))

# Create the lollipop stems
plt.vlines(x=state_avg_bod.index, ymin=0, ymax=state_avg_bod['BOD (mg/L) - Max'], color='blue', alpha=0.7, linewidth=2)

# Create the lollipop markers
plt.scatter(x=state_avg_bod.index, y=state_avg_bod['BOD (mg/L) - Max'], color='red', s=100, alpha=0.7)

# Step 3: Add a title to the plot
plt.title('Ranked Average Maximum BOD by State (Lollipop Plot)')

# Step 4: Add labels to the x and y axes
plt.xlabel('State Ranking (Lowest to Highest Average BOD)')
plt.ylabel('Average Maximum BOD (mg/L)')

# Add state names as x-axis labels for better context
plt.xticks(state_avg_bod.index, state_avg_bod['State Name'], rotation=90, ha='right')

# Step 5: Add a grid to the plot
plt.grid(True, linestyle='--', alpha=0.6)

# Step 6: Adjust the plot layout to prevent labels from overlapping
plt.tight_layout()

# Step 7: Display the plot
plt.show()

## **6. Generate descriptive statistics table**

*  **Subtask:** Calculate the requested descriptive statistics (Avg. Max pH, Avg. Max BOD, Avg. Max Conductivity, Avg. Max NitrateN, Avg. Max Total Coliform, and No. of Stations) for each state and display them in a table.


In [ ]:
# Step 1 & 2: Select relevant columns and group by 'State Name'
# Step 3 & 4: Calculate mean for specified columns and count the number of stations
state_stats = df_cleaned.groupby('State Name').agg({
    'pH - Max': 'mean',
    'BOD (mg/L) - Max': 'mean',
    'Conductivity (¬µmho/cm) - Max': 'mean',
    'NitrateN (mg/L) - Max': 'mean',
    'Total Coliform (MPN/100ml) - Max': 'mean',
    'STN code': 'count' # Count the number of stations using 'STN code'
})

# Step 6: Rename columns for clarity
state_stats = state_stats.rename(columns={
    'pH - Max': 'Avg. Max pH',
    'BOD (mg/L) - Max': 'Avg. Max BOD (mg/L)',
    'Conductivity (¬µmho/cm) - Max': 'Avg. Max Conductivity (µmho/cm)',
    'NitrateN (mg/L) - Max': 'Avg. Max NitrateN (mg/L)',
    'Total Coliform (MPN/100ml) - Max': 'Avg. Max Total Coliform (MPN/100ml)',
    'STN code': 'No. of Stations'
})

# Step 7: Display the resulting DataFrame
print("State-wise Descriptive Statistics:")
display(state_stats)

## **Summary:**
---
### Data Analysis Key Findings

*   The average BOD varies significantly across different water body types, with STP (Sewage Treatment Plant) showing the highest average BOD and WATER TREATMENT PLANT (RAW WATER) showing the lowest.
*   States were successfully ranked based on their average maximum BOD.
*   A comprehensive table of state-wise descriptive statistics was generated, including the average maximum values for pH, BOD, Conductivity, NitrateN, and Total Coliform, as well as the number of monitoring stations per state.

### Insights or Next Steps

*   Further investigation into the specific water body types with the highest average BOD (like STP) could help identify potential sources of pollution and inform targeted intervention strategies.
*   Analyzing the relationship between the ranked average maximum BOD by state and other water quality parameters in the descriptive statistics table could provide insights into potential correlations and contributing factors to high BOD levels in certain states.
